# Reddit Sentiment Analysis with RoBERTa

End-to-end NLP pipeline that classifies sentiment of Reddit posts using a pre-trained RoBERTa model (`cardiffnlp/twitter-roberta-base-sentiment-latest`).

**Topics analyzed:** e-commerce, Korean tech, logistics, supply chain

## Pipeline
1. Load Reddit posts (sample CSV or live PRAW collection)
2. Preprocess text — clean markdown, URLs, special characters
3. RoBERTa inference — Positive / Neutral / Negative with confidence scores
4. Analysis — distribution, subreddit comparison, temporal trend, keyword extraction
5. Visualization — interactive Plotly charts

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import Counter
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1. Load Data

Uses `data/real_reddit_posts.csv` — **563 real posts** collected from 7 subreddits via Reddit's public JSON API.

| Subreddit | Posts |
|---|---|
| r/ecommerce | 99 |
| r/investing | 100 |
| r/MachineLearning | 98 |
| r/startups | 100 |
| r/supplychain | 74 |
| r/korea | 54 |
| r/logistics | 38 |

To re-collect fresh data: `python collect_data.py` (no API key required)


In [ ]:
# Real Reddit posts collected via Reddit JSON API (no credentials required)
# 563 posts across 7 subreddits: ecommerce, korea, logistics, investing,
# MachineLearning, supplychain, startups
#
# To re-collect fresh data: python collect_data.py
# To use PRAW (more posts, custom subreddits): set REDDIT_CLIENT_ID/SECRET env vars first

DATA_PATH = 'data/real_reddit_posts.csv'


## 2. Text Preprocessing

Reddit posts contain markdown syntax, URLs, and formatting noise that degrades model performance. We clean before inference.

In [ ]:
def clean_reddit_text(text: str) -> str:
    """Clean Reddit markdown and noise for NLP input."""
    if not isinstance(text, str) or not text.strip():
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    # Remove markdown formatting
    text = re.sub(r'[*_~`#>|]', ' ', text)
    # Remove subreddit / user mentions
    text = re.sub(r'r/\w+|u/\w+', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def combine_title_text(row) -> str:
    """Combine title and post body; title carries more weight."""
    title = clean_reddit_text(row['title'])
    body  = clean_reddit_text(row['text'])
    if body:
        return f"{title}. {body}"
    return title


df['clean_text'] = df.apply(combine_title_text, axis=1)

# Preview
print('Before:', df['title'].iloc[0])
print('After :', df['clean_text'].iloc[0])

## 3. Load RoBERTa Model

**Model:** `cardiffnlp/twitter-roberta-base-sentiment-latest`  
A RoBERTa-base model fine-tuned on ~124M tweets for 3-class sentiment (Positive / Neutral / Negative).  
Social media text (casual, short, opinionated) closely matches Reddit post style.

In [ ]:
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

LABELS = [model.config.id2label[i] for i in range(len(model.config.id2label))]
print(f'Labels: {LABELS}')
print('Model ready.')

## 4. Run Inference

In [ ]:
def predict_batch(texts: list, batch_size: int = 16, max_length: int = 128):
    """Run RoBERTa inference on a list of texts. Returns labels and confidence scores."""
    all_labels, all_scores = [], []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length,
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**inputs).logits

        probs = F.softmax(logits, dim=-1).cpu()
        pred_ids = probs.argmax(dim=-1).tolist()

        all_labels.extend([model.config.id2label[p] for p in pred_ids])
        all_scores.extend(probs.max(dim=-1).values.tolist())

        if (i // batch_size + 1) % 5 == 0:
            print(f'  {i + batch_size}/{len(texts)} done')

    return all_labels, all_scores


print('Running RoBERTa inference ...')
texts = df['clean_text'].tolist()
labels, scores = predict_batch(texts)

df['sentiment']   = labels
df['confidence']  = scores

# Numeric encoding for correlation analysis
SENTIMENT_NUM = {'negative': -1, 'neutral': 0, 'positive': 1}
df['sentiment_score'] = df['sentiment'].map(SENTIMENT_NUM)

print(f'\nDone. {len(df)} posts labeled.')
df[['title', 'subreddit', 'sentiment', 'confidence']].head(10)

## 5. Sentiment Distribution

In [ ]:
PALETTE = {'positive': '#38A169', 'neutral': '#718096', 'negative': '#E53E3E'}

# --- Overall distribution ---
dist = df['sentiment'].value_counts().reset_index()
dist.columns = ['sentiment', 'count']

fig_dist = px.pie(
    dist, names='sentiment', values='count',
    color='sentiment', color_discrete_map=PALETTE,
    title='Overall Sentiment Distribution',
    hole=0.4
)
fig_dist.update_traces(textinfo='percent+label')
fig_dist.show()

print(dist.to_string(index=False))

In [ ]:
# --- By subreddit ---
sub_sent = df.groupby(['subreddit', 'sentiment']).size().reset_index(name='count')

fig_sub = px.bar(
    sub_sent, x='subreddit', y='count', color='sentiment',
    color_discrete_map=PALETTE, barmode='stack',
    title='Sentiment by Subreddit',
    labels={'count': 'Number of Posts', 'subreddit': 'Subreddit'}
)
fig_sub.show()

## 6. Temporal Sentiment Trend

In [ ]:
df['week'] = df['created_utc'].dt.to_period('W').apply(lambda p: p.start_time)
weekly = df.groupby(['week', 'sentiment']).size().reset_index(name='count')

fig_trend = px.line(
    weekly, x='week', y='count', color='sentiment',
    color_discrete_map=PALETTE,
    title='Weekly Sentiment Trend',
    markers=True,
    labels={'week': 'Week', 'count': 'Posts'}
)
fig_trend.show()

# Rolling mean sentiment score
df_sorted = df.sort_values('created_utc')
df_sorted['rolling_sentiment'] = df_sorted['sentiment_score'].rolling(window=7, min_periods=1).mean()

fig_roll = px.line(
    df_sorted, x='created_utc', y='rolling_sentiment',
    title='7-Post Rolling Mean Sentiment Score (-1 = Negative, +1 = Positive)',
    labels={'created_utc': 'Date', 'rolling_sentiment': 'Sentiment Score'}
)
fig_roll.add_hline(y=0, line_dash='dash', line_color='gray')
fig_roll.show()

## 7. Top Posts by Sentiment

In [ ]:
def top_posts(sentiment_label, n=5):
    subset = df[df['sentiment'] == sentiment_label].nlargest(n, 'score')
    print(f"\n{'='*60}")
    print(f"Top {n} {sentiment_label.upper()} posts (by Reddit score)")
    print('='*60)
    for _, row in subset.iterrows():
        print(f"[r/{row['subreddit']}] score={row['score']:,}  conf={row['confidence']:.2f}")
        print(f"  {row['title']}")

top_posts('positive')
top_posts('negative')
top_posts('neutral')

## 8. Confidence Analysis

In [ ]:
fig_conf = px.box(
    df, x='sentiment', y='confidence',
    color='sentiment', color_discrete_map=PALETTE,
    title='Model Confidence by Sentiment Class',
    points='all'
)
fig_conf.show()

print('\nMean confidence per class:')
print(df.groupby('sentiment')['confidence'].mean().round(3).to_string())

## 9. Upvote Score vs Sentiment

In [ ]:
fig_scatter = px.scatter(
    df, x='score', y='confidence',
    color='sentiment', color_discrete_map=PALETTE,
    size='num_comments',
    hover_data=['title', 'subreddit'],
    title='Reddit Score vs Model Confidence (bubble = comment count)',
    labels={'score': 'Reddit Upvotes', 'confidence': 'Model Confidence'}
)
fig_scatter.show()

# Correlation
corr = df[['score', 'num_comments', 'sentiment_score', 'confidence']].corr()
print('\nCorrelation matrix:')
print(corr.round(3).to_string())

## 10. Keyword Analysis by Sentiment

Most frequent words in each sentiment class (after removing stopwords).

In [ ]:
STOPWORDS = set([
    'the','a','an','is','it','in','on','at','to','and','or','but','of',
    'for','with','this','that','i','you','we','they','he','she','my','your',
    'its','are','was','been','be','have','has','had','do','does','did',
    'not','so','as','by','from','than','more','just','all','also','can',
    'get','got','one','will','would','their','which','what','how','now',
    'up','out','about','if','even','still','like','s','re','ve','t','m'
])

def extract_keywords(texts, top_n=20):
    words = []
    for text in texts:
        tokens = re.findall(r"[a-zA-Z]{3,}", text.lower())
        words.extend([w for w in tokens if w not in STOPWORDS])
    return Counter(words).most_common(top_n)


fig_kw = make_subplots(rows=1, cols=3,
    subplot_titles=['Positive Keywords', 'Neutral Keywords', 'Negative Keywords'])

for col, label in enumerate(['positive', 'neutral', 'negative'], 1):
    subset_texts = df[df['sentiment'] == label]['clean_text'].tolist()
    kw = extract_keywords(subset_texts, top_n=15)
    words_list = [w for w, _ in kw]
    counts = [c for _, c in kw]

    fig_kw.add_trace(
        go.Bar(x=counts[::-1], y=words_list[::-1], orientation='h',
               marker_color=list(PALETTE.values())[col-1], name=label),
        row=1, col=col
    )

fig_kw.update_layout(title='Top Keywords by Sentiment Class', height=500, showlegend=False)
fig_kw.show()

## 11. Summary

In [ ]:
print('=' * 50)
print('SENTIMENT ANALYSIS SUMMARY')
print('=' * 50)
print(f"Total posts analyzed : {len(df)}")
print(f"Model                : {MODEL_NAME}")
print(f"Device               : {DEVICE}")
print()

for label in ['positive', 'neutral', 'negative']:
    subset = df[df['sentiment'] == label]
    print(f"{label.capitalize():10s}: {len(subset):3d} posts "
          f"({len(subset)/len(df)*100:.1f}%)  "
          f"avg confidence={subset['confidence'].mean():.3f}  "
          f"avg score={subset['score'].mean():.0f}")

print()
print(f"Overall sentiment score (mean): {df['sentiment_score'].mean():.3f}")
most_positive_sub = df.groupby('subreddit')['sentiment_score'].mean().idxmax()
most_negative_sub = df.groupby('subreddit')['sentiment_score'].mean().idxmin()
print(f"Most positive subreddit : r/{most_positive_sub}")
print(f"Most negative subreddit : r/{most_negative_sub}")